# 02 — Agent Planning

*Level 5 — Agentic RAG*

## Objective
Have the agent sketch a plan *before* acting, then run the full ReAct-style loop (`agents/rag_agent.py`) and see the plan next to what it actually did — planning is a stated intention, not a script the loop is bound to follow exactly.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "tools", "planning"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from agent_planner import make_plan

question = "A sophomore is a student in which year of a US college?"
plan = make_plan(question)
for i, step in enumerate(plan, 1):
    print(f"{i}. {step}")


1. Use graph_search to find information about the typical academic years for US colleges
2. Use web_search to verify and gather more specific information about the term "sophomore" in US college contexts
3. Use vector_search to compare the gathered information with existing knowledge bases or databases to determine the correct year


## Run the full agent and compare the plan to its actual steps


In [3]:
from agentic_common.dataset import prepare
from agentic_common.retrieval import DenseRetriever
from vector_tool import VectorTool, GetDocumentTool
from sql_tool import SqlTool
from agents.rag_agent import RAGAgent

data = prepare()
corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)

tools = {
    "vector_search": VectorTool(retriever, data.corpus),
    "get_document": GetDocumentTool(data),
    "sql_query": SqlTool(),
}
agent = RAGAgent(tools=tools, max_steps=4)
state = agent.run(question)

print("Planned:")
for step in state.plan:
    print(" -", step)
print("\nActually did:")
for call in state.tool_history:
    print(f"  {call.tool}({call.tool_input!r})")
print(f"\nStop reason: {state.stop_reason}")
print(f"Answer: {state.answer}")


Planned:
 - Use vector_search to find relevant documents related to "US college sophomore"
 - Get_document to extract information from the top-ranked document about the typical years associated with being a sophomore
 - Verify and refine the result using web_search or graph_search if necessary

Actually did:
  vector_search('"US college sophomore"')

Stop reason: sufficient_evidence
Answer: A sophomore is a student in the second year of a US college.


## What I observed

The plan often names more steps or tools than the agent actually ends up using — e.g. planning to cross-check with `sql_query` for a question the first `vector_search` call already answers well enough. That's expected, not a bug: the plan is the agent's *prior belief* about what it might need, and the sufficiency check in the loop can end the search earlier once real evidence makes the rest of the plan unnecessary.

## Next

[03 — Iterative Retrieval Loop](./03_iterative_retrieval_loop.ipynb)
